# EDA for the notes

The idea currently is to use a target encoder for the notes and join it to the tabular data, for this we need to show two things

1. The data is majorly just categorical
2. There is strong segmentation between the labels

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
notes = pd.read_csv("data/train/train_notes.csv")
tab = pd.read_csv("data/train/train_tabular.csv", usecols=["flight_id", "failure_within_horizon", "drone_id"])
full = pd.merge( left = notes, right = tab, on = "flight_id", how = "inner")

## Categorical Strings

First we show that the data has only a few categories, and that these categories are the same between test and train

In [ ]:
# Number of categories
len(full["maintenance_note"].value_counts())

23

In [ ]:
# Symmetric difference between train and test maintenance notes
test = pd.read_csv("data/test/test_notes.csv")
set(full.maintenance_note.unique()) ^ set(test.maintenance_note.unique())

set()

## Segmentation

As you can see there are 3 major categories within these labels, which makes them a good fit for target encoding

In [ ]:
# The failure rates by maintenance note, with a minimum of 20 samples per note
failure_rates = (
    full.groupby('maintenance_note')
    .failure_within_horizon
    .agg(['mean', 'count'])
    .query('count > 20')
    .assign(failure_rate=lambda x: x['mean'] * 100)
    .sort_values('mean')
    .reset_index()
)
failure_rates['zone'] = pd.cut(
    failure_rates['mean'],
    bins=[0, 0.07, 0.50, 1.0],
    labels=['routine', 'early_warning', 'critical']
)

In [ ]:
import plotly.express as px
px.bar(failure_rates, x='failure_rate', y='maintenance_note', 
       color='zone', orientation='h',
       color_discrete_map={
           'routine': '#1baf7a',
           'early_warning': '#eda100',
           'critical': '#e34948'
       })